# Exception Handling in Python

Goal: understand types of errors/exceptions, how to handle them, and nested exception handling.

## 1. Errors vs Exceptions

- **Syntax errors**: caught by the parser before the code runs (e.g. missing colon, mismatched parentheses). Python won't even start executing the file.
- **Exceptions**: errors detected during execution. If not handled, they stop the program and print a traceback.

All built-in exceptions inherit from the `BaseException` class. Most exceptions we work with inherit from `Exception` (a subclass of `BaseException`), which is why we normally catch `Exception` rather than `BaseException` — the latter also covers `SystemExit`, `KeyboardInterrupt`, and `GeneratorExit`, which usually should propagate.

## 2. Common Built-in Exception Types

| Exception | When it happens |
|---|---|
| `ValueError` | Right type, wrong value (e.g. `int("abc")`) |
| `TypeError` | Operation applied to an incompatible type |
| `KeyError` | Missing dictionary key |
| `IndexError` | Sequence index out of range |
| `AttributeError` | Attribute/method doesn't exist on an object |
| `ZeroDivisionError` | Division or modulo by zero |
| `FileNotFoundError` | File path doesn't exist |
| `NameError` | Using a name that hasn't been defined |
| `ImportError` / `ModuleNotFoundError` | Import fails |
| `StopIteration` | Iterator has no more items |

Let's trigger a few of these to see the exception type and message.

In [1]:
samples = [
    lambda: int("abc"),           # ValueError
    lambda: "a" + 1,              # TypeError
    lambda: {"x": 1}["y"],        # KeyError
    lambda: [1, 2, 3][10],        # IndexError
    lambda: (None).missing,       # AttributeError
    lambda: 1 / 0,                # ZeroDivisionError
]

for fn in samples:
    try:
        fn()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

ValueError: invalid literal for int() with base 10: 'abc'
TypeError: can only concatenate str (not "int") to str
KeyError: 'y'
IndexError: list index out of range
AttributeError: 'NoneType' object has no attribute 'missing'
ZeroDivisionError: division by zero


## 3. Basic Handling: `try` / `except`

Wrap risky code in `try`. If an exception occurs, control jumps to the matching `except` block instead of crashing the program.

In [2]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero, returning None")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
Cannot divide by zero, returning None
None


## 4. Multiple `except` Blocks and Catching Several Types

- Order matters: put more specific exceptions before more general ones (`Exception` last), since Python checks them top to bottom and uses the first match.
- A single `except` can catch multiple types using a tuple.

In [3]:
def parse_and_lookup(data: dict, key: str, index: str):
    try:
        value = data[key]
        pos = int(index)
        return value[pos]
    except KeyError:
        print(f"Key '{key}' not found")
    except ValueError:
        print(f"'{index}' is not a valid integer index")
    except (IndexError, TypeError) as e:
        print(f"Bad index/type: {type(e).__name__}: {e}")
    except Exception as e:
        # generic catch-all, placed last on purpose
        print(f"Unexpected error: {type(e).__name__}: {e}")

d = {"nums": [10, 20, 30]}
parse_and_lookup(d, "nums", "1")     # works fine
parse_and_lookup(d, "missing", "0")  # KeyError
parse_and_lookup(d, "nums", "x")     # ValueError
parse_and_lookup(d, "nums", "99")    # IndexError

Key 'missing' not found
'x' is not a valid integer index
Bad index/type: IndexError: list index out of range


## 5. `else` and `finally`

- `else`: runs only if the `try` block did **not** raise — useful to separate "risky" code from "what to do on success".
- `finally`: always runs, whether or not an exception occurred (even if the exception isn't caught, or there's a `return`) — ideal for cleanup like closing files/connections.

In [4]:
def read_number(text):
    try:
        value = int(text)
    except ValueError:
        print(f"'{text}' is not a number")
    else:
        print(f"Parsed successfully: {value}")
    finally:
        print("Done attempting to parse\n")

read_number("42")
read_number("oops")

Parsed successfully: 42
Done attempting to parse

'oops' is not a number
Done attempting to parse



## 6. Raising Exceptions and Custom Exception Classes

Use `raise` to signal an error condition yourself. Custom exceptions (subclassing `Exception`) make error handling more specific and self-documenting than reusing generic types like `ValueError` everywhere.

In [5]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the available balance."""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw {amount}, balance is only {balance}")


class Account:
    def __init__(self, balance=0):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        return self.balance


acc = Account(100)
try:
    acc.withdraw(250)
except InsufficientFundsError as e:
    print(f"Withdrawal failed: {e}")
    print(f"Details -> balance: {e.balance}, requested: {e.amount}")

Withdrawal failed: Cannot withdraw 250, balance is only 100
Details -> balance: 100, requested: 250


## 6b. Custom Exceptions with Error Codes

Attaching a stable **error code** (and optionally other structured details) to a custom exception makes it easy for calling code — or an API layer, logger, or monitoring system — to branch on the error programmatically without parsing the message string. A common pattern:

- Define an error-code enum (or plain constants).
- Give the base exception a `code` attribute, defaulting per subclass.
- Optionally add a `to_dict()` for serializing into API responses/logs.

In [6]:
from enum import Enum


class ErrorCode(Enum):
    UNKNOWN = "E000"
    NOT_FOUND = "E404"
    VALIDATION_FAILED = "E422"
    INSUFFICIENT_FUNDS = "E409"


class AppError(Exception):
    """Base class for application errors that carry a machine-readable code."""
    code = ErrorCode.UNKNOWN

    def __init__(self, message: str, code: ErrorCode = None, **details):
        super().__init__(message)
        self.message = message
        self.code = code or self.code
        self.details = details

    def to_dict(self):
        return {
            "error_code": self.code.value,
            "message": self.message,
            "details": self.details,
        }

    def __str__(self):
        return f"[{self.code.value}] {self.message}"


class NotFoundError(AppError):
    code = ErrorCode.NOT_FOUND


class ValidationError(AppError):
    code = ErrorCode.VALIDATION_FAILED


class InsufficientFundsError(AppError):
    code = ErrorCode.INSUFFICIENT_FUNDS

    def __init__(self, balance, amount):
        super().__init__(
            f"Cannot withdraw {amount}, balance is only {balance}",
            balance=balance,
            amount=amount,
        )


def withdraw(balance, amount):
    if not isinstance(amount, (int, float)):
        raise ValidationError("amount must be a number", field="amount", got=type(amount).__name__)
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount


for balance, amount in [(100, 40), (100, 250), (100, "50")]:
    try:
        print(f"New balance: {withdraw(balance, amount)}")
    except AppError as e:
        # Dispatch on the error code, same as you might on an HTTP status
        if e.code is ErrorCode.INSUFFICIENT_FUNDS:
            print(f"Handled as funds issue -> {e.to_dict()}")
        else:
            print(f"Handled generically -> {e.to_dict()}")

New balance: 60
Handled as funds issue -> {'error_code': 'E409', 'message': 'Cannot withdraw 250, balance is only 100', 'details': {'balance': 100, 'amount': 250}}
Handled generically -> {'error_code': 'E422', 'message': 'amount must be a number', 'details': {'field': 'amount', 'got': 'str'}}


## 7. Exception Chaining with `raise ... from ...`

When you catch one exception and raise a different, more meaningful one, use `from` to preserve the original cause in the traceback (`__cause__`) instead of hiding it.

In [7]:
class ConfigError(Exception):
    pass


def load_setting(config: dict, key: str):
    try:
        return int(config[key])
    except KeyError as e:
        raise ConfigError(f"Missing required setting '{key}'") from e
    except ValueError as e:
        raise ConfigError(f"Setting '{key}' must be an integer") from e


try:
    load_setting({"timeout": "abc"}, "timeout")
except ConfigError as e:
    print(f"{e}")
    print(f"Caused by: {type(e.__cause__).__name__}: {e.__cause__}")

Setting 'timeout' must be an integer
Caused by: ValueError: invalid literal for int() with base 10: 'abc'


## 8. Nested Exception Handling

You can put a `try`/`except` inside another `try` block (or inside another `except`/`finally` block). This is useful when an inner operation needs its own specific recovery logic, while the outer block handles broader failures.

Key points:
- An exception raised inside an inner `try` is first offered to the inner `except` clauses; only if none match does it propagate to the outer block.
- The inner `finally` still runs before the exception (if unhandled) escapes to the outer level.
- Avoid nesting too deeply — more than 2 levels usually means the logic should be split into separate functions instead.

In [8]:
def process_records(records: list):
    results = []
    try:
        for i, record in enumerate(records):
            # Inner try: handle a single record's own errors, keep the loop going
            try:
                value = int(record["amount"])
                if value < 0:
                    raise ValueError("amount cannot be negative")
                results.append(value)
                print(f"Record {i}: OK -> {value}")
            except KeyError:
                print(f"Record {i}: missing 'amount' field, skipping")
            except ValueError as e:
                print(f"Record {i}: invalid amount ({e}), skipping")
    except TypeError as e:
        # Outer try: catches structural problems with `records` itself,
        # e.g. it isn't iterable or an element isn't a dict
        print(f"Could not process records at all: {e}")
    finally:
        print("Finished processing batch")
    return results


data = [
    {"amount": "100"},
    {"amount": "-5"},
    {"note": "no amount here"},
    {"amount": "abc"},
    {"amount": "42"},
]

print(process_records(data))
print()
print(process_records(None))  # triggers the outer except (TypeError: not iterable)

Record 0: OK -> 100
Record 1: invalid amount (amount cannot be negative), skipping
Record 2: missing 'amount' field, skipping
Record 3: invalid amount (invalid literal for int() with base 10: 'abc'), skipping
Record 4: OK -> 42
Finished processing batch
[100, 42]

Could not process records at all: 'NoneType' object is not iterable
Finished processing batch
[]


In [9]:
# If the inner except doesn't match, the exception escalates to the outer try.
def divide_after_lookup(data: dict, key: str, divisor: int):
    try:
        print("Outer try: starting")
        try:
            print("Inner try: looking up key")
            value = data[key]          # may raise KeyError -> handled inside
            result = value / divisor   # may raise ZeroDivisionError -> NOT handled inside
        except KeyError:
            print("Inner except: key missing, defaulting to 0")
            value = 0
            result = value / divisor
        print(f"Inner block finished, result={result}")
    except ZeroDivisionError:
        print("Outer except: caught ZeroDivisionError that escaped the inner block")
        result = None
    finally:
        print("Outer finally: cleanup runs regardless")
    return result

print(divide_after_lookup({"n": 10}, "n", 2))
print()
print(divide_after_lookup({"n": 10}, "missing", 0))

Outer try: starting
Inner try: looking up key
Inner block finished, result=5.0
Outer finally: cleanup runs regardless
5.0

Outer try: starting
Inner try: looking up key
Inner except: key missing, defaulting to 0
Outer except: caught ZeroDivisionError that escaped the inner block
Outer finally: cleanup runs regardless
None
